# Step 11.1 — Why Separate Query, Key, and Value：为什么同一个表示要拆成三种角色？

Step 10 已经完成从 recurrent Seq2Seq 到 Attention 的历史桥接：一个 representation 可以根据当前需要，直接从一组其他 representations 中检索相关信息。

进入 Self-Attention 后，问题变成：

> **同一条 sequence 里的每个 position，既可能主动寻找信息，也可能被其他 positions 匹配，还可能在被选中后贡献内容。**

这三种职责就是 Query / Key / Value。

本节只回答“它们分别是什么、为什么要分开”。不写具体 Q/K/V projection equations，也不计算 Query–Key score；这些属于 Step 11.2 及以后。


## 1. Self-Attention 的起点：同一条 Sequence 中的 T 个 Representations

假设一层 Self-Attention 的输入是：

$$
X\in\mathbb R^{B\times T\times C}.
$$

其中：

```text
B = batch size
T = sequence length
C = 当前每个 position 的 representation width
```

对某个样本，可以把 sequence 看成：

$$
x_0,x_1,\ldots,x_{T-1}.
$$

每个：

$$
x_i\in\mathbb R^C.
$$

Self-Attention 的 `self` 指的是：**提出查找请求的 positions，与被查找的候选 positions 来自同一条 sequence representation set。**

但这并不意味着 Query、Key、Value 三者必须数值完全相同。

后续会让同一个 `x_i` 产生三个 role-specific views（角色专用视图）：

```text
x_i
├─ Query role
├─ Key role
└─ Value role
```

本节先理解角色，下一节才学习这些 views 怎样通过 learned projections 得到。


## 2. Query、Key、Value 到底分别负责什么？

最稳妥的理解不是背字母，而是把一次 retrieval（检索）拆成三个问题。

### Query：我现在在找什么？

**Query（查询）**属于“发起检索”的 position。

它表达：

> 当前 position 为了更新自己的 representation，需要寻找什么类型的信息？

因此 Query 影响“我想匹配谁”。

### Key：我为什么值得被你匹配？

**Key（键 / 可匹配索引）**属于候选 position。

它表达：

> 如果别人正在查找某类信息，我有哪些特征可以用于判断我是否相关？

因此 Key 主要参与 relevance / compatibility matching（相关性匹配）。

### Value：如果你选中我，我真正提供什么内容？

**Value（值 / 被取回内容）**同样属于候选 position。

它表达：

> 如果这个 position 得到了较高 attention weight，最终应该把什么 information contribution 送给查询位置？

所以可以压缩成：

```text
Query = what am I looking for?
Key   = why should I match you?
Value = what content do I return if selected?
```

Key 和 Value 来自同一个候选 position，但职责不同：一个负责**寻址 / 匹配**，一个负责**内容传递**。


## 3. 一个语言直觉：代词 `it` 找 Antecedent 时，匹配特征和取回内容不是同一件事

考虑：

```text
The animal did not cross the street because it was tired.
```

为了直觉，假设当前 position 是 `it`。

`it` 作为 Query 可能在表达：

```text
我需要一个可能的 antecedent（先行词）
最好是 singular
最好是 animate
并且和当前语境有合理关系
```

候选 `animal` 的 Key 可以突出：

```text
singular
animate entity
noun / referable entity
```

候选 `street` 的 Key 则可能突出：

```text
singular
inanimate place/object
```

因此 `animal` 更容易与当前 Query 匹配。

但如果 `animal` 被高权重选中，真正应该传给 `it` 的 Value 不需要只包含“singular / animate”这些用于寻址的特征；Value 可以携带更丰富的上下文语义。

所以：

> **用于判断“该不该选你”的特征（Key），与“选中后你实际贡献什么”的内容（Value），没有理由必须完全绑定为同一个表示。**

这个例子只是角色直觉，不代表 Attention 自动具有人类语言学规则，也不代表 attention weight 等同于确定的指代解释。


## 4. 为什么不直接让 Query / Key / Value 都等于原始 X？——可以，但更受限制

一个最简单的 Attention 当然可以设想：

```text
Query representation = x_i
Key representation   = x_j
Value representation = x_j
```

数学上这不是“不允许”。所以正确说法不是：

> “不分 Q/K/V 就无法做 Attention。”

而是：

> **让同一个 representation 同时承担三种职责，会把三个不同学习目标绑在同一个 feature space 里，限制模型的灵活性。**

具体有三个耦合。

### A. Asking features 与 Being-found features 被绑在一起

一个 position 主动寻找信息时最有用的特征，不一定和它作为候选被别人找到时最有用的特征一样。

Query 与 Key 分开后，模型可以分别学习：

```text
作为 requester，我应该强调哪些 features？
作为 candidate，我应该暴露哪些 matching features？
```

### B. Addressing features 与 Payload features 被绑在一起

Key 只需要帮助决定“相关不相关”；Value 则决定“高权重以后传什么”。

如果 Key 和 Value 被强制完全相同，用于 matching 的表示空间与真正被混合进输出的内容空间就被绑在一起。

### C. 同一个 token 在不同角色下可能需要不同视角

同一个 position 在 Self-Attention 中既可能为自己提出 Query，也同时作为其他 positions 的 Key / Value candidate。

拆开角色后，模型可以让同一个 base representation 在三种任务中突出不同信息。

这就是 roadmap 所说的：

> **one representation serving three roles is restrictive**。

关键词是 restrictive（受限制），不是 impossible（不可能）。


In [ ]:
import mlx.core as mx

# Manually chosen role-specific views for intuition only.
# These are NOT produced by learned projections in Step 11.1.

# Three sequence positions, each with two toy features.
X = mx.array([
    [1.0, 0.2],
    [0.3, 1.0],
    [0.8, 0.7],
])  # (T,C)

Q_demo = mx.array([
    [0.9, 0.1],
    [0.2, 0.8],
    [1.0, 0.0],
])

K_demo = mx.array([
    [0.8, 0.2],
    [0.1, 0.9],
    [0.6, 0.4],
])

V_demo = mx.array([
    [1.0, 0.5],
    [0.4, 1.2],
    [1.5, 0.8],
])

print('X shape     :', X.shape)
print('Q demo shape:', Q_demo.shape)
print('K demo shape:', K_demo.shape)
print('V demo shape:', V_demo.shape)

# The point is role separation, not score computation.
# No Query-Key score or attention output is computed here.


## 5. `Self` 不等于 Query / Key / Value 数值相同：再和 Cross-Attention 对照

这是非常重要的术语边界。

### Self-Attention

Query / Key / Value 的 base representations 都来自同一条 sequence：

```text
same X
├─ Query view
├─ Key view
└─ Value view
```

`self` 描述的是**来源集合相同**，不是三个 role-specific vectors 必须相同。

### Cross-Attention

Step 10 的 encoder–decoder Attention 用现代视角可以描述为：

```text
Query base representations  <- decoder side
Key base representations    <- encoder side
Value base representations  <- encoder side
```

这里 Query 与 Key/Value 的来源集合不同，因此叫 cross-attention。

这个对照进一步说明，Q/K/V 首先是**功能角色**，不是三个神秘固定对象。

### 为什么 learned projections 会建立 task-specific lookup spaces？

下一节会让模型从同一个 `X` 学出不同 Query / Key / Value views。

直觉上等于允许模型分别学习：

```text
Query space : 什么特征适合表达“我需要什么”
Key space   : 什么特征适合表达“我能否与你匹配”
Value space : 什么特征适合表达“被选中后我贡献什么”
```

这就是 role separation 的真正价值。

但本节不写 projection matrices，也不讨论具体 projection shape；这些全部留给 Step 11.2。

### Step 11.1 材料边界

- `X=(B,T,C)` 是当前 sequence positions 的 base representations。
- Query 表示当前 position 想寻找什么；Key 表示候选 position 用什么特征参与匹配；Value 表示候选被选中后贡献什么内容。
- Key 是 addressing / matching role，Value 是 payload / content role；二者没有理由必须完全相同。
- 让原始 X 同时直接承担三种角色在数学上可以工作，但会把 asking、matching、payload 三种职责耦合在同一个 representation space。
- Self-Attention 的 `self` 指 Q/K/V 的 base source 来自同一 sequence，不表示三者数值相同。
- Cross-Attention 中 Query 可以来自 decoder，而 Key/Value 来自 encoder。
- 本轮只解释为什么要分 Q/K/V；不进入 Step 11.2，不写具体 projection equations，也不计算 Query–Key compatibility matrix。
